<a href="https://colab.research.google.com/github/Decoding-Data-Science/sharjah/blob/main/notebooks/EXTRA_Diagram_Parsing_Demo_LlamaIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsing documents with diagrams in LlamaIndex

**Document:** `Al_Noor_Building_Permit_Guide.pdf` (3 pages)

| Page | Content | Has a text layer? |
|---|---|---|
| 1 | Policy text and a fee table | Yes |
| 2 | Permit approval flow chart | No, it is a picture |
| 3 | District zoning plan | No, it is a picture |

**What we will show**
1. The default parser reads page 1 but sees almost nothing on pages 2 and 3.
2. RAG therefore cannot answer questions about the flow chart or the zoning plan.
3. We add one step: Gemini looks at each diagram and writes a description.
4. We index the descriptions, and the same questions now work.

## Step 0. API key

You need **one** key for this notebook: a Gemini API key from Google AI Studio.

1. Open https://aistudio.google.com/apikey and create a key.
2. In Colab, click the key icon in the left sidebar (Secrets).
3. Add a secret named `GOOGLE_API_KEY`, paste the key, and switch on notebook access.

Never paste the key into a code cell. LlamaIndex itself is open source and needs no key.

In [1]:
%pip install -q llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai google-genai pymupdf

In [4]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Key loaded:", os.environ["GEMINI_API_KEY"][:4] + "...")

Key loaded: AQ.A...


In [5]:
# Models. Use the same ones as your Day 2 notebook.
CHAT_MODEL = "gemini-3.5-flash-lite"      # answers questions AND reads the diagrams
EMBED_MODEL = "gemini-embedding-001"      # turns text into vectors

from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model=CHAT_MODEL)
Settings.embed_model = GoogleGenAIEmbedding(model_name=EMBED_MODEL)

## Step 1. Upload the PDF

In [6]:
from google.colab import files

os.makedirs("data", exist_ok=True)
uploaded = files.upload()                       # choose Al_Noor_Building_Permit_Guide.pdf
for name in uploaded:
    os.replace(name, f"data/{name}")
PDF_PATH = "data/" + list(uploaded)[0]
print("Saved:", PDF_PATH)

Saving Al_Noor_Building_Permit_Guide.pdf to Al_Noor_Building_Permit_Guide.pdf
Saved: data/Al_Noor_Building_Permit_Guide.pdf


## Step 2. Parse with the default reader

`SimpleDirectoryReader` reads the **text layer** of the PDF, one Document per page.
Look at how many characters it finds on each page.

In [7]:
from llama_index.core import SimpleDirectoryReader

text_docs = SimpleDirectoryReader(input_files=[PDF_PATH]).load_data()

for d in text_docs:
    page = d.metadata.get("page_label")
    print(f"--- Page {page}: {len(d.text)} characters ---")
    print(d.text[:300].strip())
    print()

--- Page None: 347975 characters ---
%PDF-1.4
% ReportLab Generated PDF document (opensource)
1 0 obj
<<
/F1 2 0 R /F2 3 0 R
>>
endobj
2 0 obj
<<
/BaseFont /Helvetica /Encoding /WinAnsiEncoding /Name /F1 /Subtype /Type1 /Type /Font
>>
endobj
3 0 obj
<<
/BaseFont /Helvetica-Bold /Encoding /WinAnsiEncoding /Name /F2 /Subtype /Type1 /Type



**Notice:** page 1 has plenty of text. Pages 2 and 3 only have the heading, the caption and the footer.
The flow chart and the zoning plan are invisible to the parser.

## Step 3. Ask questions (before)

In [8]:
from llama_index.core import VectorStoreIndex

QUESTIONS = [
    "What is the application fee for a commercial building?",                    # page 1 (text)
    "How many working days does the Planning Committee review take?",            # page 2 (flow chart)
    "What happens if the plot area is above 5,000 sqm?",                         # page 2 (flow chart)
    "Which block is directly north of Central Park, and what is its maximum number of floors?",  # page 3 (zoning plan)
    "What is located directly west of Central Park?",                            # page 3 (zoning plan)
]

def ask_all(index, label):
    engine = index.as_query_engine(similarity_top_k=3)
    print(f"================ {label} ================")
    for q in QUESTIONS:
        print("Q:", q)
        print("A:", str(engine.query(q)).strip())
        print()

index_before = VectorStoreIndex.from_documents(text_docs)
ask_all(index_before, "BEFORE: text layer only")

KeyboardInterrupt: 

Only the first question can be answered. The other four answers live inside pictures.

## Step 4. Let Gemini read the diagrams

For every page that contains a picture, we:
1. render the page as an image,
2. send it to Gemini with a clear instruction,
3. keep the written description as a new Document.

In [9]:
import pymupdf                   # PyMuPDF: renders PDF pages as images
from google import genai
from google.genai import types
from llama_index.core import Document

client = genai.Client()          # uses GOOGLE_API_KEY

DIAGRAM_PROMPT = """You are preparing this page for a search system.
Describe the diagram on this page completely and precisely in plain text.
- If it is a flow chart: list every step in order, every decision, each Yes/No branch
  and where it leads, and every number or time limit shown.
- If it is a map or zoning plan: list every block with its label, zone code, use and
  limits. Then state, for each block, what lies directly north, south, east and west
  of it. Include street names and the legend.
Do not guess. Only describe what is visible."""

diagram_docs = []
pdf = pymupdf.open(PDF_PATH)
for page_number, page in enumerate(pdf, start=1):
    if not page.get_images():            # no picture on this page, skip it
        continue
    png_bytes = page.get_pixmap(dpi=150).tobytes("png")
    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=[types.Part.from_bytes(data=png_bytes, mime_type="image/png"), DIAGRAM_PROMPT],
    )
    diagram_docs.append(Document(
        text=response.text,
        metadata={"file_name": os.path.basename(PDF_PATH), "page_label": str(page_number),
                  "content_type": "diagram_description"},
    ))
    print(f"--- Page {page_number}: description written by Gemini ---")
    print(response.text)
    print()

--- Page 2: description written by Gemini ---
Based on the image provided, here is a complete and precise description of the flowchart titled "5. Permit approval process":

### Overview of Steps and Flow

1. **Applicant submits application through the online portal** (Dark blue rectangle at the top)
   - Leads down to:

2. **Completeness check (2 working days)** (Teal rectangle)
   - Leads down to a decision diamond: **Documents complete?**

3. **Decision: Documents complete?**
   - **No branch:** Leads right to **Returned to applicant (resubmit within 10 days)** (Red rectangle). From there, an arrow loops back up to re-enter the "Completeness check" step.
   - **Yes branch:** Leads down to the next decision diamond: **Plot area above 5,000 sqm?**

4. **Decision: Plot area above 5,000 sqm?**
   - **No branch:** Leads down and left to **Technical review (7 working days)** (Teal rectangle).
   - **Yes branch:** Leads down and right to **Planning Committee review (15 working days)** (Teal

**Read the descriptions above before going on.** This is the most important habit in document AI:
always inspect what the parser produced. If the description is wrong, every answer will be wrong.

## Step 5. Ask the same questions (after)

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# Larger chunks so one diagram description stays together
index_after = VectorStoreIndex.from_documents(
    text_docs + diagram_docs,
    transformations=[SentenceSplitter(chunk_size=1024, chunk_overlap=100)],
)
ask_all(index_after, "AFTER: text + diagram descriptions")

## Expected answers (check the model against these)

| Question | Correct answer |
|---|---|
| Application fee for a commercial building | AED 2,000 |
| Planning Committee review time | 15 working days |
| Plot area above 5,000 sqm | Goes to Planning Committee review instead of Technical review (7 working days) |
| Block directly north of Central Park | Block B, Residential R2, maximum 4 floors |
| Directly west of Central Park | Block D, the School |

## What we learned

- **Parsing decides what the model can see.** No text layer means no retrieval.
- **Diagrams must be turned into words** (or handled by a multimodal parser) before indexing.
- **Inspect parsed output** before you build the index.
- **Vision is not perfect.** It reads flow charts well. For detailed maps, measurements and scale, use the real data (GIS layers, tables), not the picture.

## Optional: LlamaParse (second API key)

LlamaParse is LlamaIndex's managed cloud parser. It does the OCR and diagram step for you, but it needs a
separate `LLAMA_CLOUD_API_KEY` from https://cloud.llamaindex.ai and your document is uploaded to their service.
Check your organisation's data policy before using it with real documents.